# Tutorial 9 - Curve Fitting

---

## 1. Fitting vs Interpolation

| Property | Interpolation | Curve Fitting |
|----------|--------------|---------------|
| Data passes through? | **Yes**, exactly | **No** (in general) |
| Number of parameters | = number of data points | << number of data points |
| Goal | Reconstruct at intermediate pts | Find underlying model |
| When to use | Dense, clean data | Sparse, noisy data |

**Curve fitting** seeks model parameters $\theta$ that minimize a goodness-of-fit criterion:

$$\min_{\theta} \sum_{i=1}^{m} \left(y_i - f(x_i; \theta)\right)^2 = \|\mathbf{r}\|_2^2$$

where the **residuals** are $r_i = y_i - f(x_i; \theta)$. This is **ordinary least squares (OLS)**.

In [1]:
import sys, math, random
sys.path.insert(0, '../')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from numethods.fitting import PolyFit, LinearFit, ExpFit, NonlinearFit, plot_fit, plot_residuals
from numethods.linalg import Matrix, Vector
from numethods.orthogonal import LeastSquaresSolver

random.seed(42)
print("numethods fitting module loaded.")

numethods fitting module loaded.


## 2. Linear Least Squares: The Normal Equations

For a model **linear in parameters** $\theta = (\theta_1, \ldots, \theta_k)^T$:
$$y_i = \theta_1 \phi_1(x_i) + \theta_2 \phi_2(x_i) + \cdots + \theta_k \phi_k(x_i) + \varepsilon_i$$

we minimize $\|A\theta - \mathbf{y}\|_2^2$ where the **design matrix** is:
$$A_{ij} = \phi_j(x_i)$$

The solution satisfies the **normal equations**:
$$A^T A \theta = A^T \mathbf{y}$$

However, `numethods` solves this via **QR decomposition** of $A$ (more numerically stable than forming $A^T A$):
$$A = QR \quad \Rightarrow \quad R \theta = Q^T \mathbf{y}$$

In [2]:
# Polynomial fit: basis functions phi_j(x) = x^j
# Example: fit y = a + bx + cx^2 to noisy quadratic data

def true_quad(x):
    return 2.0 - 3.0*x + 1.5*x**2

x_data = [i*0.5 for i in range(11)]  # 0, 0.5, ..., 5
y_data = [true_quad(xi) + random.gauss(0, 0.5) for xi in x_data]

# Fit degree 1, 2, 3, 4
fits = {}
for deg in [1, 2, 3, 4]:
    fits[deg] = PolyFit(x_data, y_data, degree=deg)

# Print degree-2 coefficients
fit2 = fits[2]
print("Polynomial Fit (degree 2):")
fit2.summary()
print(f"True coefficients: c0=2.0, c1=-3.0, c2=1.5")

# Evaluate fits
x_fine = [i*0.05 for i in range(101)]
y_true_fine = [true_quad(xi) for xi in x_fine]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(x_data, y_data, color='black', zorder=5, s=50, label='Noisy data')
axes[0].plot(x_fine, y_true_fine, 'k--', linewidth=1.5, label='True quadratic')
colors = ['blue', 'red', 'green', 'purple']
for deg, col in zip([1, 2, 3, 4], colors):
    y_fit = [fits[deg].evaluate(xi) for xi in x_fine]
    # compute MSE
    mse = sum((fits[deg].evaluate(xi) - yi)**2 for xi,yi in zip(x_data, y_data)) / len(x_data)
    axes[0].plot(x_fine, y_fit, '-', color=col, linewidth=1.5, label=f'Degree {deg} (MSE={mse:.3f})')
axes[0].set_title('Polynomial Fits to Noisy Quadratic Data')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Residuals for degree 2
res2 = [yi - fits[2].evaluate(xi) for xi, yi in zip(x_data, y_data)]
axes[1].bar(x_data, res2, width=0.3, color='steelblue', alpha=0.8)
axes[1].axhline(0, color='k', linewidth=1)
axes[1].set_title('Residuals: Degree-2 Fit')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y - p(x)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('polyfit.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved polyfit.png")

Polynomial Fit (degree 2):
Polynomial Fit Coefficients
degree = 2
 coeff |   value
-------------------
  c0  |  2.022713
  c1  | -3.126621
  c2  |  1.527581

True coefficients: c0=2.0, c1=-3.0, c2=1.5
Saved polyfit.png


![alt text](polyfit.png)

## 3. Linear Fit with Custom Basis Functions

`LinearFit` allows any basis functions $\{\phi_j(x)\}$, not just monomials. The model is still **linear in parameters**:

$$f(x; \mathbf{c}) = \sum_j c_j \phi_j(x)$$

### Examples of useful basis sets
| Application | Basis functions |
|-------------|----------------|
| Polynomial | $1, x, x^2, \ldots$ |
| Fourier | $1, \cos(\omega x), \sin(\omega x), \ldots$ |
| Physics-based | $1/r, \log r, r^2, \ldots$ |
| Splines | Piecewise polynomials |
| Radial basis | $e^{-\|x-c_k\|^2}$

In [3]:
# Example: fit gravity anomaly signature
# True: g(x) = a * z / (x^2 + z^2)  (Euler-type)
# Basis: phi_1(x) = 1/((x-x0)^2+1), phi_2(x) = x/((x-x0)^2+1)

xs = [i*10.0 for i in range(-10, 11)]   # -100 to 100 m
z_true = 50.0   # depth
A_true = 5.0    # amplitude
def gz_true(x): return A_true * z_true / (x**2 + z_true**2)
ys_gz = [gz_true(xi) + random.gauss(0, 0.02) for xi in xs]

# Fit with Euler-type basis
z0_guess = 50.0  # if we know the source depth
phi1 = lambda x: z0_guess / (x**2 + z0_guess**2)
phi2 = lambda x: x / (x**2 + z0_guess**2)
phi3 = lambda x: 1.0  # constant offset

basis = [phi1, phi2, phi3]
lfit = LinearFit(xs, ys_gz, basis)
lfit.summary()

y_fit_gravity = [lfit.evaluate(xi) for xi in xs]
residuals = [yi - lfit.evaluate(xi) for xi, yi in zip(xs, ys_gz)]
rms = math.sqrt(sum(r**2 for r in residuals)/len(residuals))

print(f"RMS residual: {rms:.4f}")
print(f"True amplitude parameter: {A_true:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(xs, ys_gz, color='black', s=30, label='Observed', zorder=5)
ax1.plot(xs, [gz_true(xi) for xi in xs], 'k--', linewidth=1.5, label='True')
ax1.plot(xs, y_fit_gravity, 'r-', linewidth=2, label=f'Linear fit (RMS={rms:.3f})')
ax1.set_title('Custom Basis Fit: Gravity Anomaly'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.set_xlabel('x (m)'); ax1.invert_yaxis()

ax2.plot(xs, residuals, 'b-o', markersize=4)
ax2.axhline(0, color='k'); ax2.grid(True, alpha=0.3)
ax2.set_title('Residuals'); ax2.set_xlabel('x (m)'); ax2.set_ylabel('Residual')

plt.tight_layout()
plt.savefig('custom_basis_fit.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved custom_basis_fit.png")

Linear Fit Coefficients
 basis |   value
-------------------
  φ0  |  4.919373
  φ1  | -0.162450
  φ2  |  0.001729

RMS residual: 0.0159
True amplitude parameter: 5.000
Saved custom_basis_fit.png


![alt text](custom_basis_fit.png)

## 4. Exponential Fit via Linearization

For $y \approx a e^{bx}$, take the logarithm:
$$\ln y = \ln a + b x = c_0 + c_1 x$$

This is a **linear** problem in $(c_0, c_1)$! Solve it, then $a = e^{c_0}$, $b = c_1$.

⚠️ **Warning:** The log transform **changes the error structure**. Minimizing $\|\ln y_i - \ln f(x_i)\|^2$ is **not** the same as minimizing $\|y_i - f(x_i)\|^2$. For noisy data with heteroscedastic errors, `NonlinearFit` is more appropriate.

In [4]:
# Generate exponential-looking data
x_exp = [i*0.5 for i in range(9)]  # 0 to 4
A_true, B_true = 3.0, 0.7
y_exp = [A_true * math.exp(B_true * xi) * (1 + random.gauss(0, 0.05))
         for xi in x_exp]

ef = ExpFit(x_exp, y_exp)
ef.summary()
ef.trace()

print(f"True parameters: a={A_true}, b={B_true}")
print(f"Fitted:          a={ef.a:.4f}, b={ef.b:.4f}")

Exponential Fit Parameters
 param |   value
-------------------
   a   |  2.821590
   b   |  0.718451

Exponential Fit Trace (log transform)
 x | y | log(y)
-------------------
 0.0000 |  2.6070 |  0.9582
 0.5000 |  4.1280 |  1.4178
 1.0000 |  5.7646 |  1.7517
 1.5000 |  8.9485 |  2.1915
 2.0000 |  12.5697 |  2.5313
 2.5000 |  16.2115 |  2.7857
 3.0000 |  25.5365 |  3.2401
 3.5000 |  33.0230 |  3.4972
 4.0000 |  49.1212 |  3.8943

True parameters: a=3.0, b=0.7
Fitted:          a=2.8216, b=0.7185


## 5. Nonlinear Least Squares: Levenberg-Marquardt

For models **nonlinear in parameters** (e.g., $f(x; a, b) = a \sin(bx + c)$), we need an iterative method.

### Gauss–Newton Algorithm
Linearize the residuals around current estimate $\theta^{(k)}$:
$$r_i(\theta) \approx r_i(\theta^{(k)}) + J_{ij} \delta_j, \quad J_{ij} = \frac{\partial r_i}{\partial \theta_j}$$

This yields a linear least-squares problem:
$$J^T J \, \delta = -J^T \mathbf{r}$$

### Levenberg–Marquardt (LM)
Adds a damping term $\lambda I$ to improve stability and convergence:
$$(J^T J + \lambda I) \, \delta = -J^T \mathbf{r}$$

- If $\lambda$ large: gradient descent (stable but slow)
- If $\lambda$ small: Gauss–Newton (fast convergence near solution)
- Adaptively adjusts $\lambda$ based on whether the step improves the fit

`numethods.NonlinearFit` implements adaptive LM with numerical Jacobian (choice of `forward`, `backward`, `central`, `central4th` differentiation).

In [5]:
# Nonlinear fit: sinusoidal with unknown amplitude, frequency, phase
def sin_model(x, params):
    A, omega, phi = params
    return A * math.sin(omega * x + phi)

# True parameters
A0, omega0, phi0 = 2.5, 1.3, 0.5
x_sin = [i*0.4 for i in range(16)]
y_sin = [sin_model(xi, [A0, omega0, phi0]) + random.gauss(0, 0.15)
         for xi in x_sin]

# Fit with initial guess
init_params = [2.0, 1.0, 0.0]
nf = NonlinearFit(sin_model, x_sin, y_sin, init_params,
                  max_iter=100, tol=1e-8, lam=1e-3, verbose=False)

print(f"True params:   A={A0}, ω={omega0}, φ={phi0}")
print(f"Fitted params: A={nf.params[0]:.4f}, ω={nf.params[1]:.4f}, φ={nf.params[2]:.4f}")
print(f"Converged in {len(nf.history)} iterations")

# Visualize
x_fine_s = [i*0.1 for i in range(65)]
y_true_s = [sin_model(xi, [A0, omega0, phi0]) for xi in x_fine_s]
y_fit_s = [nf.evaluate(xi) for xi in x_fine_s]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.scatter(x_sin, y_sin, color='black', s=40, zorder=5, label='Noisy data')
ax1.plot(x_fine_s, y_true_s, 'k--', linewidth=1.5, label='True curve')
ax1.plot(x_fine_s, y_fit_s, 'r-', linewidth=2, label='LM fit')
ax1.set_title('Nonlinear Least Squares: Sinusoidal Fit')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_xlabel('x')

# Convergence of residual norm
res_norms = [h[2] for h in nf.history]
ax2.semilogy(range(len(res_norms)), res_norms, 'b-o', markersize=4)
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Residual norm (log scale)')
ax2.set_title('LM Convergence'); ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('nonlinear_fit.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved nonlinear_fit.png")

True params:   A=2.5, ω=1.3, φ=0.5
Fitted params: A=2.5522, ω=1.3123, φ=0.4616
Converged in 16 iterations
Saved nonlinear_fit.png


![alt text](nonlinear_fit.png)

## 6. Model Selection: Overfitting vs Underfitting

Adding more parameters always reduces the training error, but can make the model worse on new data - **overfitting**.

### Key metrics
- **Residual Sum of Squares (RSS)**: $\sum(y_i - f(x_i))^2$
- **Coefficient of determination** $R^2$: $1 - \text{RSS}/\text{TSS}$ (1 = perfect fit)
- **Akaike Information Criterion (AIC)**: $2k - 2\ln(L)$ (penalizes complexity)
- **BIC**: $k\ln(n) - 2\ln(L)$ (stronger penalty for large $n$)

Lower AIC/BIC → better model (balancing fit and parsimony).

In [6]:
# Underfitting vs overfitting demonstration
# True: y = sin(2*pi*x), x in [0,1], 12 noisy samples

def true_f(x): return math.sin(2*math.pi*x)

n_pts = 12
x_pts = [(i+0.5)/n_pts for i in range(n_pts)]
y_pts = [true_f(xi) + random.gauss(0, 0.2) for xi in x_pts]

# Fit degrees 1 through 11
degrees = list(range(1, 12))
rss_vals = []
r2_vals  = []
aic_vals = []

y_mean = sum(y_pts)/len(y_pts)
tss = sum((yi - y_mean)**2 for yi in y_pts)

for deg in degrees:
    pf = PolyFit(x_pts, y_pts, degree=deg)
    residuals = [yi - pf.evaluate(xi) for xi, yi in zip(x_pts, y_pts)]
    rss = sum(r**2 for r in residuals)
    r2 = 1 - rss/tss
    k = deg + 1  # number of parameters
    # AIC (for OLS with known variance): n*ln(RSS/n) + 2k
    n_ = len(x_pts)
    sigma2 = rss/n_ if rss > 0 else 1e-30
    aic = n_ * math.log(sigma2) + 2*k
    
    rss_vals.append(rss)
    r2_vals.append(r2)
    aic_vals.append(aic)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Model Selection: Overfitting vs Underfitting")

axes[0].plot(degrees, rss_vals, 'b-o', markersize=6)
axes[0].set_xlabel('Polynomial Degree'); axes[0].set_ylabel('RSS')
axes[0].set_title('Residual Sum of Squares\n(always decreases with degree)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(degrees, r2_vals, 'g-o', markersize=6)
axes[1].set_xlabel('Polynomial Degree'); axes[1].set_ylabel('R²')
axes[1].set_title('R² (also always increases\n— not a reliable selector)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(degrees, aic_vals, 'r-o', markersize=6)
best_deg = degrees[aic_vals.index(min(aic_vals))]
axes[2].axvline(best_deg, color='k', linestyle='--', label=f'Best: degree {best_deg}')
axes[2].set_xlabel('Polynomial Degree'); axes[2].set_ylabel('AIC')
axes[2].set_title(f'AIC (minimum at degree {best_deg}\n— selects appropriate complexity)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_selection.png', dpi=100, bbox_inches='tight')
plt.close()

print("AIC values and best model:")
for d, a in zip(degrees, aic_vals):
    marker = "  ← best" if d == best_deg else ""
    print(f"  Degree {d:2d}: AIC = {a:.3f}{marker}")

AIC values and best model:
  Degree  1: AIC = -14.229
  Degree  2: AIC = -12.467
  Degree  3: AIC = -35.409
  Degree  4: AIC = -34.026
  Degree  5: AIC = -35.190
  Degree  6: AIC = -33.953
  Degree  7: AIC = -36.330
  Degree  8: AIC = -35.170
  Degree  9: AIC = -46.142
  Degree 10: AIC = -66.488
  Degree 11: AIC = -464.114  ← best


![alt text](model_selection.png)

## 7. Application: Velocity-Depth Gradient Fitting

In seismic refraction, we often fit a **velocity–depth model** to observed travel-time data. A common model is:

$$V(z) = V_0 + k \cdot z$$

where $V_0$ is the surface velocity and $k$ is the gradient (velocity increases with depth). This is a linear fit in $(V_0, k)$.

In [7]:
# Simulated velocity measurements at boreholes (depth, velocity)
depths = [0, 100, 250, 400, 600, 800, 1000, 1300, 1600, 2000]  # meters
V0_true, k_true = 1800.0, 0.6   # m/s, (m/s)/m

velocities = [V0_true + k_true * z + random.gauss(0, 30) for z in depths]

# Linear fit: V = V0 + k*z
# Basis: [1, z]
basis_linear = [lambda z: 1.0, lambda z: z]
lf_vel = LinearFit(depths, velocities, basis_linear)

print("Velocity–Depth Linear Fit:")
print(f"  V₀ (surface velocity) = {lf_vel.coeffs[0]:.1f} m/s  (true: {V0_true})")
print(f"  k  (velocity gradient)  = {lf_vel.coeffs[1]:.3f} (m/s)/m  (true: {k_true})")

# Also try quadratic (geologically unrealistic — shows overfitting)
basis_quad = [lambda z: 1.0, lambda z: z, lambda z: z**2]
lf_quad = LinearFit(depths, velocities, basis_quad)

# Compute R² for each
def r2_score(observed, predicted):
    mean_obs = sum(observed)/len(observed)
    ss_tot = sum((o-mean_obs)**2 for o in observed)
    ss_res = sum((o-p)**2 for o,p in zip(observed, predicted))
    return 1 - ss_res/ss_tot

pred_lin = [lf_vel.evaluate(z) for z in depths]
pred_quad = [lf_quad.evaluate(z) for z in depths]
print(f"\nLinear fit R² = {r2_score(velocities, pred_lin):.6f}")
print(f"Quadratic fit R² = {r2_score(velocities, pred_quad):.6f}")
print("(Higher R² ≠ better model when comparing different complexity models!)")

# Plot
z_fine = [i*20 for i in range(101)]
v_true_fine = [V0_true + k_true * z for z in z_fine]
v_lin_fine = [lf_vel.evaluate(z) for z in z_fine]
v_quad_fine = [lf_quad.evaluate(z) for z in z_fine]

fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(velocities, depths, color='black', s=60, zorder=5, label='Borehole data')
ax.plot(v_true_fine, z_fine, 'k--', linewidth=1.5, label='True V(z)')
ax.plot(v_lin_fine, z_fine, 'b-', linewidth=2, label=f'Linear fit')
ax.plot(v_quad_fine, z_fine, 'r:', linewidth=1.5, label='Quadratic fit')
ax.invert_yaxis()
ax.set_xlabel('Velocity (m/s)'); ax.set_ylabel('Depth (m)')
ax.set_title('Seismic Velocity–Depth Model\n(Linear Regression)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('velocity_depth_fit.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved velocity_depth_fit.png")

Velocity–Depth Linear Fit:
  V₀ (surface velocity) = 1823.5 m/s  (true: 1800.0)
  k  (velocity gradient)  = 0.580 (m/s)/m  (true: 0.6)

Linear fit R² = 0.995437
Quadratic fit R² = 0.996014
(Higher R² ≠ better model when comparing different complexity models!)
Saved velocity_depth_fit.png


![alt text](velocity_depth_fit.png)

## 8. Summary

| Class | Model | Parameters | Method |
|-------|-------|-----------|--------|
| `PolyFit(x, y, degree)` | $\sum c_j x^j$ | $c_0, \ldots, c_d$ | QR least squares |
| `LinearFit(x, y, basis)` | $\sum c_j \phi_j(x)$ | $c_0, \ldots, c_k$ | QR least squares |
| `ExpFit(x, y)` | $a e^{bx}$ | $a, b$ | Log-linearization |
| `NonlinearFit(model, x, y, p0)` | any $f(x;\theta)$ | $\theta$ | Levenberg–Marquardt |

### Comparison with interpolation
- Polynomial interpolation through $n+1$ points gives **exact fit** (oscillation risk!)
- Polynomial least squares with degree $d << n$ gives **smooth fit** (noise reduction)
- `PolyFit` is **not** an interpolating polynomial, it minimizes squared residuals

### Exercises
1. Compare `ExpFit` and `NonlinearFit` on exponential data with 10% multiplicative noise. Which is more accurate?
2. Fit the model $y = a \cdot \log(bx)$ using `NonlinearFit`. What's a good initial guess?
3. For the velocity–depth problem, add a sinusoidal layer perturbation to the true model. How many free parameters do you need? Can `LinearFit` handle it?
4. Implement cross-validation manually: split the data into train/test sets, fit polynomial degrees 1–8, and plot test MSE vs degree. At what degree does overfitting begin?
5. Compute the **95% confidence intervals** on the linear fit parameters using the formula $\text{Var}(\hat{c}) = \sigma^2 (A^T A)^{-1}$, where $\sigma^2$ is estimated from residuals.